# 2차시 실습 — 프롬프트 · 컨텍스트 · 정식 프로젝트 승격

**같은 모델이라도 '어떻게 묻고 무엇을 보여주느냐'가 품질을 바꾼다**

## 🧪 이 노트북 사용법

- 이 노트북은 워크샵 **실습(Codex CLI)의 실행 가능한 동반 자료**입니다. 각자 발급받은 **`OPENAI_API_KEY`** 로 바로 돌아갑니다.
- 실제 캠프에서는 **터미널의 Codex CLI**로 같은 작업을 합니다. 그래서 각 단계마다
  - **▶︎ 노트북에서 실행** (파이썬 셀) 과
  - **⌨️ 터미널 Codex 명령** (복붙용) 을 함께 적어 두었습니다.
- 위에서부터 **순서대로** 셀을 실행하세요 (`Shift+Enter`).
- 황금률: **작게 요청 → 확인 → 수정** (1차시 iteration loop).

> ⚠️ API 호출에는 약간의 토큰 비용이 듭니다. 기본 모델은 저렴한 `gpt-4o-mini`.

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 실행
%pip install -q openai
import os, getpass, re, pathlib, json
from openai import OpenAI
from IPython.display import IFrame, Markdown, display

# 각자 발급받은 OPENAI_API_KEY 입력 (입력값은 화면에 보이지 않습니다)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY를 붙여넣고 Enter: ")

client = OpenAI()
MODEL = "gpt-4o-mini"   # 품질을 더 원하면 "gpt-4o" 또는 "gpt-4.1" 로 바꾸세요

def ask(prompt, system=None, **kw):
    """프롬프트 하나를 보내고 답변 텍스트를 돌려준다."""
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, messages=msgs, **kw)
    return r.choices[0].message.content

def show(text):
    display(Markdown(text))

print("준비 완료! MODEL =", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료! MODEL = gpt-4o-mini


In [2]:
# 🔧 HTML 생성·미리보기 도우미 (HTML을 만드는 차시에서 사용)
def extract_code(text):
    """답변에서 첫 번째 코드블록만 꺼낸다. 없으면 원문 그대로."""
    if "```" not in text:
        return text.strip()
    block = text.split("```", 2)[1]
    if "\n" in block:
        first, rest = block.split("\n", 1)
        if first.strip().isalpha():   # ```html 같은 언어 태그 제거
            return rest.strip()
    return block.strip()

def save_and_preview(html, filename="app.html", height=420):
    pathlib.Path(filename).write_text(html, encoding="utf-8")
    print("저장됨 →", filename, " (브라우저로 직접 열어도 됩니다)")
    return IFrame(filename, width="100%", height=height)

def iterate(current_html, instruction):
    """현재 HTML 전체를 보여주고(=컨텍스트) 한 가지 수정을 요청한다."""
    prompt = f"""아래는 현재 HTML 전체야:
```html
{current_html}
```
다음 한 가지만 반영해서 **전체 HTML을 다시** 출력해줘: {instruction}
- 설명 없이 HTML 코드만 출력해."""
    return extract_code(ask(prompt))

## 🖥️ Codex를 쓰는 곳(표면) & 이 캠프의 기준

Codex는 한 곳에만 있지 않습니다 — **같은 에이전트**를 여러 서비스에서 씁니다:

| 서비스 | 무엇을 할 수 있나 |
|---|---|
| **터미널 CLI** (`@openai/codex`) | 터미널에서 직접 코드 읽기·고치기·실행. **이 캠프의 중심** |
| **IDE 확장** (VS Code·Cursor·Windsurf) | 에디터 안에서 채팅·편집·미리보기, 이미지 드래그&드롭, UI 이미지 생성 |
| **웹 / 클라우드** (chatgpt.com/codex) | 작업을 클라우드에 위임·진행 모니터링, GitHub 이슈/PR에서 `@codex` 호출 |
| **Codex 앱(데스크톱)** | 여러 에이전트 **병렬**·worktree, 스레드 **팝아웃**, **인앱 브라우저** 미리보기, **Automations(예약 자동 실행)** |
| **ChatGPT iOS 앱** | 이동 중 모바일에서 Codex |

> 📌 **이 워크샵은 '터미널 Codex CLI' 중심으로 진행합니다.**
> 이유 ① 모든 서비스의 **공통분모** ② 설치·환경이 단순(0차시 1회면 끝) ③ CLI·IDE·앱이 **같은 설정(AGENTS.md·config)·에이전트**를 공유 → CLI를 익히면 나머지 서비스로 **바로 확장**됩니다.

**출처**: [Codex 앱](https://developers.openai.com/codex/app) · [IDE 확장](https://developers.openai.com/codex/ide) · [웹/클라우드](https://developers.openai.com/codex/cloud)

## Lab A — 좋은 프롬프트 5요소 

**Goal · User · Behavior · Constraints · Success**

아래 셀은 *나쁜 프롬프트* 와 *5요소를 담은 프롬프트* 의 결과를 나란히 비교합니다. 차이를 눈으로 확인하세요.

In [3]:
bad = "할 일 앱 만들어줘"

good = """[Goal] 할 일을 추가/완료/삭제하는 단일 HTML 페이지를 만들어줘.
[User] 메모 앱이 익숙한 일반 직장인.
[Behavior] 입력창+추가 버튼 / 항목 클릭 시 완료(취소선) / 삭제 버튼.
[Constraints] 외부 라이브러리 없이 바닐라 HTML/CSS/JS, 한국어 UI.
[Success] 새로고침 전까지 추가·완료·삭제가 모두 동작하면 완료."""

print("===== 나쁜 프롬프트 결과(요약) =====")
print(ask("다음 요청을 받으면 무엇을 만들지 3줄로만 요약해: " + bad))
print("\n===== 좋은 프롬프트 결과(요약) =====")
print(ask("다음 요청을 받으면 무엇을 만들지 3줄로만 요약해: " + good))

===== 나쁜 프롬프트 결과(요약) =====
할 일 목록을 작성하고 관리할 수 있는 앱입니다. 각 작업에 대해 마감일, 우선순위, 알림 기능을 제공합니다. 사용자는 작업을 완료할 때마다 쉽게 체크할 수 있습니다.

===== 좋은 프롬프트 결과(요약) =====
단일 HTML 페이지를 생성하여 할 일을 추가할 수 있는 입력창과 버튼을 제공합니다. 항목 클릭 시 완료 표시(취소선)와 함께 삭제 기능이 포함됩니다. 외부 라이브러리 없이 바닐라 HTML, CSS, JS로 구현하며 한국어 UI로 구성됩니다.


## Lab B — 고급 기법: Structured Output (JSON) & Few-shot 

결과를 **정해진 구조(JSON)** 로 받으면 후처리가 쉽습니다. `response_format`으로 강제할 수 있어요.

In [4]:
resp = ask(
    "점심 메뉴 6개를 추천해줘. 각 항목은 name(이름)과 cuisine(한식/양식/중식 중 하나)을 가진 "
    "JSON 객체로, {\"menus\": [...]} 형태로만 답해.",
    response_format={"type": "json_object"},
)
data = json.loads(resp)
for m in data["menus"]:
    print(f'- {m["name"]} ({m["cuisine"]})')

- 비빔밥 (한식)
- 치킨 파르메잔 (양식)
- 짬뽕 (중식)
- 불고기 (한식)
- 핫윙 (양식)
- 마파두부 (중식)


## Lab C — Chain-of-Thought & One Task at a Time 

복잡한 작업은 **단계적으로 생각하게** 하면 정확해집니다. 그리고 **한 번에 하나씩**(One Task) 요청하는 게 1차시부터 이어진 황금률.

In [5]:
puzzle = "사과 3개(개당 1200원)와 우유 2개(개당 2500원)를 사고 10000원을 냈어. 거스름돈은 얼마야?"
print("=== 바로 답 요청 ===")
print(ask(puzzle + " 한 줄로만 답해."))
print("\n=== 단계적으로 생각하게(CoT) ===")
print(ask(puzzle + " 계산 과정을 단계별로 보여주고, 마지막 줄에 '거스름돈: N원' 으로 답해."))

=== 바로 답 요청 ===
거스름돈은 2000원입니다.

=== 단계적으로 생각하게(CoT) ===
사과와 우유의 총 비용을 계산해 보겠습니다.

1. **사과의 총 비용 계산**:
   - 사과 1개의 가격: 1200원
   - 사과 3개의 가격: 3 × 1200 = 3600원

2. **우유의 총 비용 계산**:
   - 우유 1개의 가격: 2500원
   - 우유 2개의 가격: 2 × 2500 = 5000원

3. **총 비용 계산**:
   - 총 비용 = 사과의 총 비용 + 우유의 총 비용
   - 총 비용 = 3600 + 5000 = 8600원

4. **거스름돈 계산**:
   - 지불한 금액: 10000원
   - 거스름돈 = 지불한 금액 - 총 비용
   - 거스름돈 = 10000 - 8600 = 1400원

따라서, 거스름돈은 1400원입니다.

거스름돈: 1400원


## Lab D — AGENTS.md 초안 만들기 

⌨️ **터미널 Codex:** `codex "이 프로젝트용 AGENTS.md 초안을 만들어줘 (구조·한국어 주석·함수 분리)"`

In [6]:
draft = ask("1차시에서 만든 '랜덤 점심 추천기'(단일 HTML) 프로젝트를 위한 AGENTS.md 초안을 만들어줘. "
            "구조, 한국어 주석 규칙, 함수 분리 원칙, 커밋 규칙을 포함해. 마크다운으로.")
show(draft)
pathlib.Path("AGENTS.md").write_text(draft, encoding="utf-8")
print("\nAGENTS.md 저장 완료 — 팀 규칙에 맞게 직접 다듬으세요.")

# AGENTS.md

## 1. 프로젝트 구조

이 프로젝트는 단일 HTML 파일로 구성되어 있으므로, 아래와 같은 구조를 가집니다.

```
/random-lunch-suggester
│
├── index.html         # 랜덤 점심 추천기 메인 HTML 파일
│
├── style.css          # 스타일 시트를 포함하는 CSS 파일 (필요 시)
│
└── script.js          # JavaScript 로직을 포함하는 파일
```

## 2. 한국어 주석 규칙

- 모든 함수 및 복잡한 코드 블록에는 한글로 설명하는 주석을 추가합니다.
- 주석은 한 줄 또는 블록 형태로 작성할 수 있으며, 의미 있는 내용을 포함해야 합니다.

예시:
```javascript
// 랜덤 점심 메뉴를 추천하는 함수
function getRandomLunch() {
    // ...
}
```

## 3. 함수 분리 원칙

- 각 함수는 하나의 특정 작업만 수행해야 하며, 이를 통해 코드의 가독성을 높이고 유지보수를 용이하게 합니다.
- 복잡한 로직의 경우, 함수를 분리하여 재사용 가능하도록 합니다.

예시:
```javascript
// 점심 메뉴 목록을 가져오는 함수
function getLunchMenu() {
    // ...
}

// 랜덤으로 점심 메뉴를 추천하는 함수
function suggestRandomLunch() {
    const menu = getLunchMenu();
    // ...
}
```

## 4. 커밋 규칙

- 모든 커밋 메시지는 영어로 작성하되, 간결하고 명확하게 내용을 표현합니다.
- 커밋 메시지는 다음과 같은 형식을 따릅니다.
  - `feat:` 새로운 기능 추가
  - `fix:` 버그 수정
  - `docs:` 문서 변경
  - `style:` 스타일 변경 (포맷 규칙 수정)
  - `refactor:` 코드 리팩토링
  - `test:` 테스트 추가 또는 수정
  - `chore:` 빌드 업무 수정 또는 패키지 매니저 변경

예시:
```
feat: Add random lunch suggestion functionality
fix: Correct typo in lunch menu array
docs: Update README with usage instructions
```

---

이 문서는 '랜덤 점심 추천기' 프로젝트를 위한 기본 개발 규칙을 정리한 것입니다. 모든 팀원은 이 규칙을 준수하여 협업 효율성을 높이고, 코드 품질을 유지하도록 합니다.


AGENTS.md 저장 완료 — 팀 규칙에 맞게 직접 다듬으세요.


## Lab E — 에러를 컨텍스트로 (디버깅) 

막혔을 때의 정석: **에러 메시지를 요약하지 말고 원문 그대로** + 관련 코드를 함께 준다.

In [7]:
broken = """function pickLunch(menus){
  return menus[Math.floor(Math.random() * menus.lenght)];  // 어딘가 이상함
}
console.log(pickLunch(["김밥", "파스타", "초밥"]));"""
error = "가끔 결과가 undefined 로 나옵니다. (브라우저 콘솔: TypeError 관련)"

show(ask(f"""아래 코드와 증상을 보고 (1) 원인 (2) 고친 전체 코드 (3) 한 줄 설명 을 줘.

[코드]
```
{broken}
```
[증상/에러 — 원문 그대로]
{error}"""))

(1) 원인: 코드에서 `menus.length`를 잘못 작성하여 `lenght`로 오타가 있습니다. 이로 인해 `menus.lenght`는 `undefined`가 되어, 곱셈 연산의 결과가 `NaN`이 되어 `Math.floor`의 결과도 `NaN`이 되고, `menus` 배열의 인덱스로 접근하는 과정에서 `undefined`가 발생합니다.

(2) 고친 전체 코드:
```javascript
function pickLunch(menus) {
  return menus[Math.floor(Math.random() * menus.length)];  // 오타 수정: lenght -> length
}
console.log(pickLunch(["김밥", "파스타", "초밥"]));
```

(3) 한 줄 설명: `menus.length`의 오타를 수정하여 랜덤한 메뉴를 올바르게 선택하도록 수정했습니다.

# 🧰 Codex 확장 — 슬래시 명령 · 스킬 · 플러그인 · MCP 도구

> 여기부터는 **Codex CLI(터미널)** 에서 쓰는 기능 설명입니다(노트북에서 실행 X). `codex` 를 켠 상태를 기준으로 읽으세요.
> 명령·도구는 버전마다 다를 수 있으니, 항상 **입력창에서 `/` 메뉴**와 공식 문서(developers.openai.com/codex)로 최신을 확인하세요.

## Lab F — 슬래시 명령어 사전 

Codex 대화창에서 **`/`** 를 치면 명령 메뉴가 뜹니다. 자주 쓰는 것 위주로:

| 명령 | 역할 |
|---|---|
| `/model` | 사용할 AI 모델 변경 |
| `/permissions` (승인) | Codex가 **묻지 않고** 할 수 있는 범위 설정 |
| `/plan` | 바로 코딩 대신 **계획 모드**로 전환 |
| `/diff` | 방금 만든 **Git 변경·파일 수정** 내용 보기 |
| `/review` | 현재 변경분에 대한 **코드 리뷰** 요청 |
| `/mention` | 특정 **파일을 대화에 첨부** |
| `/skills` | **스킬** 목록 보기·적용 |
| `/mcp` | 연결된 **MCP 도구** 목록 확인 |
| `/plugins` · `/apps` | **플러그인·앱(커넥터)** 관리·연결 |
| `/compact` | 대화를 요약해 **토큰 절약** |
| `/clear` · `/new` | 새 대화 시작 |
| `/resume` · `/fork` | 세션 이어가기 · 대화 분기 |
| `/status` · `/usage` | 세션 설정 · **토큰 사용량** 확인 |
| `/import` | **Claude Code 설정 가져오기** |

> 전체 목록은 길어요. 외우지 말고 **`/` 치고 검색**하세요. 위 14개만 알아도 충분합니다.

## Lab G — 스킬 & 플러그인 

**스킬(Skill)** = Codex에게 주는 *재사용 업무 매뉴얼*. 지시문·자료·스크립트를 한 폴더에 묶어 둡니다.
- **호출**: `/skills` 메뉴에서 고르거나, 작업 설명이 맞으면 Codex가 **자동 선택**. (스킬 모드에선 `$스킬이름`)
- **위치**: 프로젝트 `.codex/skills/` · 내 계정 `~/.codex/skills/` · Codex 내장
- **만들기**: 폴더 + `SKILL.md`(필수 항목 `name`·`description`) / 또는 `$skill-creator`·녹화(Record & Replay)
- **언제 스킬?** "**정해진 절차**를 반복"할 때(예: 배포 체크리스트). **새 연결·외부 데이터**가 필요하면 스킬이 아니라 **MCP**.

**플러그인/앱** = Codex 기능을 더 붙이는 확장. `/plugins`(플러그인 관리), `/apps`(외부 커넥터 연결), `/import`(Claude Code 설정 가져오기).
> **제공되는 플러그인·확장 목록은 `/plugins` 메뉴에서 직접 둘러보세요** — 버전마다 추가/변경됩니다.

정리: **슬래시 명령**(세션 조작) · **스킬**(반복 절차) · **MCP**(외부 도구 연결) · **플러그인/앱**(기능 확장) — 4종을 구분해서 쓰면 됩니다.


> 👇 아래 **G-1 / G-2**에서 "오늘 뭐 입지?" 스킬을 직접 만들고, 플러그인으로 묶어 연결해 봅니다.

## Lab G-1 — 스킬 만들기: "오늘 뭐 입지?" (날씨 × 분위기 → 옷차림 추천) 

**정해진 절차를 반복**하는 일이니 스킬감입니다: *날씨 확인 → 분위기 확인 → 규칙대로 추천 → 고정 형식 출력*.

스킬은 결국 **폴더 + `SKILL.md`(업무 매뉴얼)** 입니다. 여기서는
1. `skills_lab/.codex/skills/outfit-stylist/SKILL.md` 를 만들고
2. 노트북에서 **Codex가 이 스킬을 쓰는 상황을 시뮬레이션**(SKILL.md를 시스템 프롬프트로 주입)해 동작을 확인한 뒤
3. 터미널 Codex에서 `$outfit-stylist` 로 실제 호출해 봅니다.


In [8]:
# ① 스킬 폴더 + SKILL.md 만들기
import pathlib

SKILL_DIR = pathlib.Path("skills_lab/.codex/skills/outfit-stylist")
SKILL_DIR.mkdir(parents=True, exist_ok=True)

SKILL_MD = """---
name: outfit-stylist
description: 오늘 날씨(기온·강수·바람)와 분위기/일정에 맞춰 옷차림을 추천한다. "뭐 입지", "옷 추천", "코디" 요청에 사용.
---

# Outfit Stylist — 오늘 뭐 입지?

## 절차
1. **날씨 확인**: 기온(현재/최고), 강수 확률, 바람. 없으면 도시를 물어 `curl \"wttr.in/<도시>?format=j1\"` 로 조회.
2. **분위기/일정 확인**: 예) 데이트, 면접, 재택+저녁 산책, 운동.
3. **추천 규칙**
   - 기온: 28°C↑ 반팔·린넨 / 20~27 얇은 긴팔 / 12~19 가디건·자켓 / 5~11 코트·니트 / 4↓ 패딩·목도리
   - 강수 50%↑ → 방수 신발·우산, 밝은색 하의 금지
   - 분위기: 데이트=포인트 컬러 1개 / 면접=차분한 톤+자켓 / 운동=기능성 소재
4. **출력 형식(고정)**
   - 한 줄 요약(오늘의 컨셉)
   - 상의 / 하의 / 겉옷 / 신발 / 소품 — 표로
   - "비 오면?" 대비 한 줄
"""

(SKILL_DIR / "SKILL.md").write_text(SKILL_MD, encoding="utf-8")
print("✅ 생성:", SKILL_DIR / "SKILL.md")
print(SKILL_MD)


✅ 생성: skills_lab/.agents/skills/outfit-stylist/SKILL.md
---
name: outfit-stylist
description: 오늘 날씨(기온·강수·바람)와 분위기/일정에 맞춰 옷차림을 추천한다. "뭐 입지", "옷 추천", "코디" 요청에 사용.
---

# Outfit Stylist — 오늘 뭐 입지?

## 절차
1. **날씨 확인**: 기온(현재/최고), 강수 확률, 바람. 없으면 도시를 물어 `curl "wttr.in/<도시>?format=j1"` 로 조회.
2. **분위기/일정 확인**: 예) 데이트, 면접, 재택+저녁 산책, 운동.
3. **추천 규칙**
   - 기온: 28°C↑ 반팔·린넨 / 20~27 얇은 긴팔 / 12~19 가디건·자켓 / 5~11 코트·니트 / 4↓ 패딩·목도리
   - 강수 50%↑ → 방수 신발·우산, 밝은색 하의 금지
   - 분위기: 데이트=포인트 컬러 1개 / 면접=차분한 톤+자켓 / 운동=기능성 소재
4. **출력 형식(고정)**
   - 한 줄 요약(오늘의 컨셉)
   - 상의 / 하의 / 겉옷 / 신발 / 소품 — 표로
   - "비 오면?" 대비 한 줄



In [ ]:
# ② 스킬 시뮬레이션 — 실제 날씨를 가져와 분위기만 바꿔 재사용
import json as _json, urllib.request

def today_weather(city="Seoul"):
    """wttr.in에서 오늘 날씨 요약(무료·키 불필요). 실패하면 직접 입력값 사용."""
    try:
        with urllib.request.urlopen(f"https://wttr.in/{city}?format=j1", timeout=5) as r:
            w = _json.load(r)
        cur, day = w["current_condition"][0], w["weather"][0]
        rain = max(int(h["chanceofrain"]) for h in day["hourly"])
        return f"{city} 현재 {cur['temp_C']}°C (최고 {day['maxtempC']}°C), 강수확률 {rain}%, 바람 {cur['windspeedKmph']}km/h"
    except Exception:
        return "서울 현재 24°C (최고 29°C), 강수확률 60%, 바람 14km/h   # 오프라인 예시값"

skill = (SKILL_DIR / "SKILL.md").read_text(encoding="utf-8")

def outfit(mood, city="Seoul"):
    w = today_weather(city)
    return ask(f"오늘 날씨: {w}\n오늘 분위기/일정: {mood}\n옷차림을 추천해줘.",
               system="너는 아래 SKILL.md 매뉴얼을 그대로 따르는 스타일리스트 스킬이다.\n\n" + skill)

show(outfit("퇴근 후 데이트"))
show(outfit("오전에 중요한 면접"))   # 같은 스킬, 분위기만 바꿔 재사용 — '매뉴얼'의 힘


## Lab G-2 — 플러그인으로 묶기: 스타일 팩 만들고 연결 

스킬이 **매뉴얼 1개**라면, **플러그인은 스킬·훅·앱 연결을 묶어 나눠 쓰는 패키지**입니다.
방금 만든 `outfit-stylist` 스킬을 **my-style-pack** 플러그인으로 포장하고, **로컬 마켓플레이스**로 등록해 설치합니다.

```
skills_lab/                                ← 로컬 마켓플레이스 루트
├── .agents/plugins/marketplace.json       # 마켓플레이스 (배포할 플러그인 목록)
└── my-style-pack/                         ← 플러그인
    ├── .codex-plugin/plugin.json          # 마니페스트 (필수: name·version)
    └── skills/outfit-stylist/SKILL.md     # G-1에서 만든 스킬
```

> 💡 이 구조를 손으로 안 짜도 됩니다 — Codex 내장 **`$plugin-creator`** 스킬이 자동 스캐폴딩해 줍니다. 여기서는 구조를 이해하기 위해 직접 만들어 봅니다.


In [ ]:
# ③ 플러그인 패키지 + 로컬 마켓플레이스 만들기
import shutil, pathlib, json as _json

PACK = pathlib.Path("skills_lab/my-style-pack")
(PACK / ".codex-plugin").mkdir(parents=True, exist_ok=True)

# 마니페스트 (.codex-plugin/plugin.json)
(PACK / ".codex-plugin" / "plugin.json").write_text(_json.dumps({
    "name": "my-style-pack",
    "version": "0.1.0",
    "description": "날씨×분위기 옷차림 추천 — outfit-stylist 스킬",
    "skills": "./skills/"
}, ensure_ascii=False, indent=2), encoding="utf-8")

# G-1 스킬을 팩 안으로 복사
shutil.copytree(SKILL_DIR, PACK / "skills" / "outfit-stylist", dirs_exist_ok=True)

# 로컬 마켓플레이스 (skills_lab을 마켓플레이스 루트로)
MKT = pathlib.Path("skills_lab/.agents/plugins")
MKT.mkdir(parents=True, exist_ok=True)
(MKT / "marketplace.json").write_text(_json.dumps({
    "name": "my-style-marketplace",
    "plugins": [{
        "name": "my-style-pack",
        "source": {"source": "local", "path": "./my-style-pack"},
        "policy": {"installation": "AVAILABLE", "authentication": "ON_INSTALL"},
        "category": "Productivity"
    }]
}, ensure_ascii=False, indent=2), encoding="utf-8")

for p in sorted(pathlib.Path("skills_lab").rglob("*")):
    if p.is_file(): print("📦", p)


### ⌨️ 터미널 Codex에서 스킬·플러그인 연결 (노트북 밖에서)

```bash
# ① 스킬을 내 계정에 설치 — 어느 프로젝트에서든 사용 가능
mkdir -p ~/.codex/skills
cp -r skills_lab/my-style-pack/skills/outfit-stylist ~/.codex/skills/

# ② Codex 실행 후 스킬 호출 (둘 다 해보기)
#    $outfit-stylist 서울 날씨 기준으로 퇴근 후 데이트 룩 추천해줘
#    "오늘 뭐 입지? 저녁에 데이트 있어"   ← 설명(description)이 맞으면 자동 선택되는지 확인

# ③ 플러그인 연결 — 로컬 마켓플레이스 등록 → /plugins 에서 설치
codex plugin marketplace add ./skills_lab   # skills_lab을 마켓플레이스로 등록
codex plugin marketplace list               # 등록 확인
codex                                       # 실행 후:
#    /plugins → my-style-marketplace 섹션 → my-style-pack → Install plugin
#    새 대화(new thread)를 시작해야 플러그인이 반영됩니다
#    @my-style-pack 오늘 서울 데이트 룩 추천해줘   ← @ 로 플러그인 지정 호출
```

**확인 포인트**: ① `/skills` 목록에 `outfit-stylist`가 보이는가 ② 자연어 요청만으로 스킬이 **자동 선택**되는가 ③ `/plugins` 설치 후 새 대화에서 `@my-style-pack`이 동작하는가.

> 💡 끄고 싶을 땐 삭제 대신 `~/.codex/config.toml` 에 `[plugins."my-style-pack"]` `enabled = false` 를 넣고 재시작하면 됩니다.


## Lab G-3 — 진짜 플러그인 설치: ponytail ("안 쓴 코드가 최고의 코드") 

**ponytail** = "가장 게으른 시니어 개발자"처럼 사고하게 만드는 플러그인.
과한 추상화·불필요한 의존성·안 시킨 기능을 막아 **코드량을 줄입니다**(실측 평균 ~54%↓ — 비용·시간도 절감).

⌨️ **설치 (터미널 Codex, 노트북 밖에서):**

```bash
codex plugin marketplace add DietrichGebert/ponytail   # ① 마켓플레이스 등록
codex                                                  # ② 실행 후:
#    /plugins → ponytail → Install plugin
#    /hooks   → 라이프사이클 훅 검토 후 신뢰   ← 새 플러그인은 훅부터 확인!
#    새 대화(new thread)를 시작해야 반영됩니다
```

**모드 3단계** — 강도를 고를 수 있습니다(기본 `full`). 전환 명령은 버전마다 다를 수 있으니 `/` 메뉴에서 확인:

| 모드 | 강도 |
|---|---|
| `lite` | 부드러운 권고 — 명백한 과설계만 지적 |
| `full` | "사다리" 강제 — 이미 있는 것·표준 라이브러리 우선, 최소 diff (기본) |
| `ultra` | 극단적 미니멀 — 한 줄로 되는 건 반드시 한 줄로 |

**같이 오는 스킬**: `@ponytail-review`(현재 diff 과설계 점검) · `@ponytail-audit`(리포 전체 스캔) · `@ponytail-gain`(절감 지표)

> 👇 아래 셀에서 Lab G-1처럼 **시뮬레이션**으로, 모드가 바뀌면 같은 요청(**피보나치 구현**)의 결과가 어떻게 달라지는지 확인합니다. 터미널에서는 설치 후 새 스레드에서 "피보나치 함수 만들어줘"를 모드 바꿔가며 직접 비교해 보세요.

In [ ]:
# ponytail 모드 시뮬레이션 — 같은 요청, 모드에 따라 코드가 얼마나 달라지나
PONYTAIL_MODES = {
    "없음(플러그인 X)": None,
    "lite":  "너는 ponytail(lite) 모드의 게으른 시니어 개발자다. "
             "명백한 과설계(불필요한 클래스·설정·예외처리)만 피하고, 그 외엔 평소대로 작성해라.",
    "full":  "너는 ponytail(full) 모드의 게으른 시니어 개발자다. 사다리 규칙을 강제한다: "
             "표준 라이브러리로 되면 그걸 쓰고, 요청하지 않은 추상화·클래스·주석·검증 코드를 금지한다. "
             "동작하는 최소한의 코드만 작성해라.",
    "ultra": "너는 ponytail(ultra) 모드의 극단적으로 게으른 시니어 개발자다. "
             "한 줄로 될 수 있으면 반드시 한 줄로 작성해라. 함수는 최대 1개, 주석·설명 금지.",
}

req = "n번째 피보나치 수를 반환하는 파이썬 코드를 작성해줘. 코드만 출력해."

for mode, style in PONYTAIL_MODES.items():
    code = extract_code(ask(req, system=style))
    print(f"━━━ ponytail {mode} — {len(code.splitlines())}줄 ━━━")
    print(code)
    print()

## Lab H — 도구 확장: MCP로 Playwright·Context7 붙이기 

**MCP** = Codex에 *손발*을 달아주는 표준 연결. 터미널에서 한 줄로 추가합니다:
```bash
# (방법1) 한 줄 추가
codex mcp add context7 -- npx -y @upstash/context7-mcp
codex mcp add playwright -- npx -y @playwright/mcp@latest
```
```toml
# (방법2) ~/.codex/config.toml 에 직접
[mcp_servers.context7]
command = "npx"
args = ["-y", "@upstash/context7-mcp"]

[mcp_servers.playwright]
command = "npx"
args = ["-y", "@playwright/mcp@latest"]
```
추가 후 `codex` 에서 **`/mcp`** 로 연결을 확인하세요.

**왜 좋은가 (각 도구):**

| 도구 | 무엇 | 왜 좋은가 |
|---|---|---|
| **Context7** | 라이브러리 **최신 공식 문서**를 실시간으로 컨텍스트에 주입 | 모델이 *옛날 API*로 잘못 짜는 걸 막음 — "최신 React/Next 문법대로" 가능 |
| **Playwright** | 에이전트가 **브라우저를 직접** 열고 클릭·입력·스크린샷 | 만든 웹앱을 Codex가 **실제로 눌러보며 테스트**·버그 재현 |

## Lab I — rtk 설치 & 동작 확인 (토큰 절약 프록시) 

**rtk**(Rust Token Killer) = `git status`·`grep`·테스트 같은 **명령 출력을 압축**해 에이전트 컨텍스트에 넣어주는 셸 프록시. 긴 세션에서 **토큰 60~90% 절감** → 비용↓·속도↑. (터미널에서 진행)

```bash
# ① 설치
# macOS
brew install rtk
# Linux (또는 Windows WSL)
curl -fsSL https://raw.githubusercontent.com/rtk-ai/rtk/master/install.sh | sh
# Windows 네이티브 — 전용 설치 파일이 없어 Rust(cargo)로 설치합니다
winget install Rustlang.Rustup && rustup default stable   # Rust가 없다면 먼저 (새 터미널로)
cargo install --git https://github.com/rtk-ai/rtk
#  ⚠️ `cargo install rtk` 금지 — crates.io의 동명이인(Rust Type Kit)이 깔립니다

# ② 설치 확인 — 3종 세트
which rtk          # 경로가 나오면 OK
rtk --version      # rtk X.Y.Z
rtk gain           # 절약 통계 화면이 뜨면 진짜 rtk (동명이인이면 여기서 실패)

# ③ 압축 체감 — 같은 명령을 그냥 vs rtk로
git status          # 원본 출력
rtk git status      # 압축된 출력 — 줄 수 차이를 비교해 보세요
rtk grep "function" .   # grep도 마찬가지
```

**Codex에 연결하기** — 한 줄이면 됩니다. Codex는 훅 대신 **AGENTS.md + RTK.md 지시문 방식**으로 연결됩니다:

```bash
rtk init -g --codex     # 전역(내 계정) 설치 — AGENTS.md·RTK.md에 rtk 사용 지시문 자동 생성
# rtk init --codex      # 이 프로젝트에만 설치하려면 -g 없이
```

> 원리: Codex가 매번 읽는 AGENTS.md에 "셸 명령은 `rtk`를 앞에 붙여 실행하라"는 규칙이 들어가는 것 — 파일을 열어 무엇이 추가됐는지 직접 확인해 보세요.

```bash
# ④ 사용되는지 체크
codex "git 상태랑 최근 커밋 3개 보여줘"   # 실행 로그에 rtk git ... 이 보이는지 관찰
rtk gain                                    # Total commands·Tokens saved 수치가 늘었는지 확인
rtk gain --history                          # 어떤 명령이 얼마나 절약했는지 이력
```

**확인 포인트**: ① `rtk gain`이 통계 화면을 띄우는가 ② 같은 명령의 출력 줄 수가 눈에 띄게 줄었는가 ③ Codex 작업 후 `rtk gain`의 saved 수치가 올라가는가.

> 💡 Claude Code를 쓴다면 `rtk init -g` (훅 자동 설치, 명령 자동 재작성), Codex는 위처럼 `rtk init -g --codex` (지시문 방식) — 에이전트별 연결 방법이 다릅니다.


## Lab J — Symphony 맛보기: 이슈 트래커가 오케스트레이터 

**Symphony** = OpenAI가 공개한 Codex **오케스트레이션 사양**(오픈소스, GitHub 1.5만+ ⭐).
에이전트를 일일이 감독하는 대신 **이슈 트래커를 상태 머신**으로 삼아, 에이전트가 스스로 작업을 가져가게 합니다.

- 한 사람이 무리 없이 감독할 수 있는 동시 세션은 **3~5개** — 그 이상은 컨텍스트 전환이 병목.
- 그래서 "감독"을 트래커에 위임: 열린 이슈마다 에이전트가 하나씩 붙는다.
- 상태 머신: `TODO / IN PROGRESS / REWORK → HUMAN REVIEW → MERGING` — 사람은 **리뷰만** 한다.
- 셋업은 **Markdown 2개**가 전부: `SPEC.md`(문제 + 의도한 해법) + `WORKFLOW.md`(워크플로 정책).

여기서는 그 **미니 버전**을 직접 만들어 봅니다:
① SPEC/WORKFLOW 작성 → ② LLM으로 이슈 분해(작업 트리·의존성) → ③ 상태 보드 한 바퀴 돌리기.

In [ ]:
# ① Symphony 셋업 — 마크다운 2개가 전부 (SPEC.md + WORKFLOW.md)
SYM = pathlib.Path("symphony_lab")
SYM.mkdir(exist_ok=True)

(SYM / "SPEC.md").write_text("""# SPEC — 점심 추천기 v2
## 문제
점심 메뉴를 못 정해 시간을 낭비한다. 지금 앱은 무작위 추천만 가능.
## 의도한 해법 (높은 수준 방향 — 경로가 아니라 목표)
- 단일 HTML(app.html) 유지, 바닐라 JS
- 추천 히스토리 저장(localStorage) / "오늘은 이거 제외" 버튼 / 한식·양식·중식 필터
## 성공 기준
- 새로고침해도 히스토리가 유지된다
- 제외한 메뉴는 오늘 다시 추천되지 않는다
""", encoding="utf-8")

(SYM / "WORKFLOW.md").write_text("""# WORKFLOW — 작업 정책
- 모든 작업은 이슈로 등록하고 보드 상태를 유지한다:
  TODO / IN PROGRESS / HUMAN REVIEW / MERGING / BACKLOG
- 의존성이 풀린 이슈부터 진행(IN PROGRESS 표시). 완료하면 증명(요약·스크린샷)을 첨부해 HUMAN REVIEW로.
- 사람 승인 후에만 MERGING. 범위 밖 개선점은 새 이슈로 등록만 한다.
""", encoding="utf-8")

print("✅ 생성:", *(p.name for p in SYM.iterdir()))
print((SYM / "SPEC.md").read_text(encoding="utf-8"))

In [ ]:
# ② SPEC → 이슈 분해(작업 트리) + ③ 상태 보드 한 바퀴
spec = (SYM / "SPEC.md").read_text(encoding="utf-8")
resp = ask(
    "아래 SPEC.md를 서로 독립적으로 진행 가능한 작업 이슈 4~6개로 분해해줘. 각 이슈는 "
    '{"id": 번호, "title": "제목", "depends_on": [선행 이슈 id 목록]} 형태로, '
    '{"issues": [...]} JSON으로만 답해.\n\n' + spec,
    response_format={"type": "json_object"},
)
issues = {it["id"]: it for it in json.loads(resp)["issues"]}

# BACKLOG = 의존성 때문에 아직 막힌 이슈, TODO = 지금 바로 집을 수 있는 이슈
done = set()
state = {i: ("TODO" if not it["depends_on"] else "BACKLOG") for i, it in issues.items()}

def board():
    for col in ["TODO", "IN PROGRESS", "HUMAN REVIEW", "MERGING", "BACKLOG", "DONE"]:
        row = " · ".join(f"#{i} {issues[i]['title']}" for i in issues if state[i] == col)
        print(f"{col:13}| {row or '-'}")

def run_agent(i):
    """실제 Symphony에선 IN PROGRESS → HUMAN REVIEW(사람 승인) → MERGING을 거친다."""
    state[i] = "DONE"; done.add(i)
    for j, it in issues.items():          # 의존성이 풀린 이슈를 BACKLOG → TODO 로
        if state[j] == "BACKLOG" and set(it["depends_on"]) <= done:
            state[j] = "TODO"

print("=== 분해 직후 ===");  board()
first = next(i for i in issues if state[i] == "TODO")
run_agent(first)
print(f"\n=== 에이전트가 #{first} 완료 후 — 막혀 있던 이슈가 풀린다 ===");  board()

### ⌨️ 터미널 Codex로 Symphony 패턴 (노트북 밖에서)

방금 만든 `symphony_lab/`에서 Codex에게 "보드 관리"까지 맡겨 봅니다:

```bash
cd symphony_lab && codex
# 프롬프트:
#   SPEC.md와 WORKFLOW.md를 읽고, 작업을 이슈로 분해해 BOARD.md
#   (TODO / IN PROGRESS / HUMAN REVIEW / MERGING / BACKLOG 보드)를 만들어줘.
#   의존성 없는 이슈 하나를 골라 구현하고, 끝나면 보드를 갱신해 HUMAN REVIEW로 옮겨줘.
```

**확인 포인트**: ① 이슈에 의존성(작업 트리)이 생겼는가 ② 에이전트가 **막히지 않은 이슈부터** 집었는가 ③ 내가 할 일이 "구현"이 아니라 **"리뷰"** 로 바뀌었는가.

> 💡 진짜 Symphony는 Linear 같은 이슈 트래커에 연결해, 열린 이슈마다 에이전트 워크스페이스를 하나씩 띄웁니다. 참조 구현은 Elixir지만 핵심은 **Markdown 사양**일 뿐 — 어떤 코딩 에이전트로든 재현 가능. 큰 교훈 두 가지: 에이전트를 경직된 노드로 다루지 말고 **"목표"를 부여**하라, 그리고 **"이슈가 곧 작업 단위"** 라는 사고방식.

## ⌨️ 터미널에서 마무리 (노트북 밖에서 진행)

이 부분은 실제 파일·Git이 필요해 **터미널 Codex**로 합니다:

```bash
# ② AGENTS.md 규칙대로 리팩터링
codex "AGENTS.md 규칙에 맞춰 app.html 코드를 리팩터링해줘"

```

> 학습 포인트: **같은 모델 + 좋은 컨텍스트 = 다른 품질.**

## ✅ 마무리 체크리스트

- [ ] Lab A: 나쁜/좋은 프롬프트 차이를 확인
- [ ] Lab B: JSON 구조 출력 + CoT 비교
- [ ] Lab D: AGENTS.md 초안 생성
- [ ] Lab E: 에러를 컨텍스트로 디버깅
- [ ] Lab F: 슬래시 명령 `/` 메뉴 둘러보기
- [ ] Lab G: 스킬·플러그인 구분 이해 + `outfit-stylist` 스킬·`my-style-pack` 플러그인 만들기
- [ ] Lab G-3: ponytail 설치 → 모드(lite/full/ultra)별 피보나치 비교
- [ ] Lab H: MCP로 Context7/Playwright 추가
- [ ] Lab I: rtk 설치 → `rtk gain`·압축 체감·Codex 연결 확인
- [ ] Lab J: Symphony 미니 보드 — SPEC 분해·상태 머신 한 바퀴
- [ ] (터미널) 리팩터링·커밋까지